<a href="https://colab.research.google.com/github/RaniellyPatricia/previsao-conversao-leads-cursos/blob/main/notebooks/02_limpeza_preparacao_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Limpeza e preparação dos dados

Este notebook faz parte do MVP 1 do projeto de Produto de Dados para Análise e Previsão de Conversão de Leads em Cursos.

Na etapa anterior, foi realizado o entendimento inicial da base, incluindo análise de linhas, colunas, tipos de dados, valores nulos, duplicidades, variável de conversão e categorias mais frequentes.

Nesta etapa, o objetivo é preparar uma versão tratada da base para análise exploratória e construção do dashboard.

Serão realizadas as seguintes ações:

- criar uma cópia da base original;
- tratar valores nulos;
- padronizar categorias;
- remover ou ignorar colunas com baixo valor analítico;
- selecionar colunas relevantes para o MVP 1;
- validar a base tratada;
- exportar o arquivo final para uso no Power BI.

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

Saving Leads_X_Education_traduzido_PTBR.csv to Leads_X_Education_traduzido_PTBR.csv


In [2]:
df = pd.read_csv("Leads_X_Education_traduzido_PTBR.csv")
df.head()

,ID do Prospect,Número do Lead,Origem do Lead,Fonte do Lead,Não Enviar E-mail,Não Ligar,Convertido,Total de Visitas,Tempo Total no Site,Visualizações de Página por Visita,...,Receber Atualizações Sobre Conteúdo de Marketing Digital,Perfil do Lead,Cidade,Índice de Atividade Assimétrica,Índice de Perfil Assimétrico,Pontuação de Atividade Assimétrica,Pontuação de Perfil Assimétrico,Concordo em Pagar o Valor por Cheque,Cópia Gratuita de Mastering The Interview,Última Atividade Relevante
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Chat Olark,Não,Não,0,0.0,0,0.0,...,Não,Não informado,Não informado,02.Médio,02.Médio,15.0,15.0,Não,Não,Modificado
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,API,Busca Orgânica,Não,Não,0,5.0,674,2.5,...,Não,Não informado,Não informado,02.Médio,02.Médio,15.0,15.0,Não,Não,E-mail Aberto
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,Envio por Landing Page,Tráfego Direto,Não,Não,1,2.0,1532,2.0,...,Não,Lead Potencial,Mumbai,02.Médio,01.Alto,14.0,20.0,Não,Sim,E-mail Aberto
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc,660719,Envio por Landing Page,Tráfego Direto,Não,Não,0,1.0,305,1.0,...,Não,Não informado,Mumbai,02.Médio,01.Alto,13.0,17.0,Não,Não,Modificado
4,3256f628-e534-4826-9d63-4a8b88782852,660681,Envio por Landing Page,Google,Não,Não,1,2.0,1428,1.0,...,Não,Não informado,Mumbai,02.Médio,01.Alto,15.0,18.0,Não,Não,Modificado


## Criação da base de trabalho

Nesta etapa, foi criada uma cópia da base original. A base original será mantida sem alterações, e todos os tratamentos serão feitos em uma nova tabela chamada `df_tratado`.

In [11]:
df_tratado = df.copy()
df_tratado.columns = (
    df_tratado.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("ç", "c")
    .str.replace("ã", "a")
    .str.replace("á", "a")
    .str.replace("à", "a")
    .str.replace("â", "a")
    .str.replace("é", "e")
    .str.replace("ê", "e")
    .str.replace("í", "i")
    .str.replace("ó", "o")
    .str.replace("ô", "o")
    .str.replace("õ", "o")
    .str.replace("ú", "u")
)
df_tratado.columns

Index(['id_do_prospect', 'numero_do_lead', 'origem_do_lead', 'fonte_do_lead',
       'nao_enviar_e-mail', 'nao_ligar', 'convertido', 'total_de_visitas',
       'tempo_total_no_site', 'visualizacoes_de_pagina_por_visita',
       'ultima_atividade', 'pais', 'especializacao',
       'como_soube_da_x_education', 'ocupacao_atual',
       'o_que_mais_importa_na_escolha_do_curso', 'busca', 'revista',
       'artigo_de_jornal', 'foruns_da_x_education', 'jornal',
       'anuncio_digital', 'por_recomendacoes',
       'receber_mais_atualizacoes_sobre_nossos_cursos', 'tags',
       'qualidade_do_lead', 'atualizar_sobre_conteudo_de_supply_chain',
       'receber_atualizacoes_sobre_conteudo_de_marketing_digital',
       'perfil_do_lead', 'cidade', 'indice_de_atividade_assimetrica',
       'indice_de_perfil_assimetrico', 'pontuacao_de_atividade_assimetrica',
       'pontuacao_de_perfil_assimetrico',
       'concordo_em_pagar_o_valor_por_cheque',
       'copia_gratuita_de_mastering_the_interview',
   

## Validação da variável de conversão

A variável de conversão foi validada para garantir que possui apenas os valores esperados: `0` para lead não convertido e `1` para lead convertido.

In [5]:
df_tratado["convertido"].value_counts(dropna=False)
df_tratado["convertido"].unique()

array([0, 1])

## Tratamento de valores nulos

Com base no diagnóstico inicial, algumas colunas relevantes possuem valores ausentes. Para manter essas colunas na análise, os valores nulos serão substituídos por `Não informado`.

Essa decisão evita perda de registros e permite visualizar a ausência de informação como uma categoria específica no dashboard.

In [18]:
colunas_preencher_nao_informado = [
    "fonte_do_lead",
    "ultima_atividade",
    "pais",
    "especializacao",
    "como_soube_da_x_education",
    "ocupacao_atual",
    "o_que_mais_importa_na_escolha_do_curso",
    "tags",
    "qualidade_do_lead",
    "perfil_do_lead",
    "cidade",
    "indice_de_atividade_assimetrica",
    "indice_de_perfil_assimetrico"
]

for coluna in colunas_preencher_nao_informado:
    if coluna in df_tratado.columns:
        df_tratado[coluna] = df_tratado[coluna].fillna("Não informado")

In [16]:
colunas_numericas_preencher_zero = [
    "total_de_visitas",
    "visualizacoes_de_pagina_por_visita",
    "pontuacao_de_atividade_assimetrica",
    "pontuacao_de_perfil_assimetrico"
]

for coluna in colunas_numericas_preencher_zero:
    if coluna in df_tratado.columns:
        df_tratado[coluna] = df_tratado[coluna].fillna(0)

In [19]:
df_tratado.isnull().sum().sort_values(ascending=False)

,0
id_do_prospect,0
numero_do_lead,0
origem_do_lead,0
fonte_do_lead,0
nao_enviar_e-mail,0
nao_ligar,0
convertido,0
total_de_visitas,0
tempo_total_no_site,0
visualizacoes_de_pagina_por_visita,0


## Remoção de colunas de baixo valor analítico

Com base no entendimento inicial e no dicionário de dados, algumas colunas não serão priorizadas no MVP 1 por apresentarem baixa variação, predominância de uma única resposta ou pouca utilidade analítica para o dashboard inicial.

Essas colunas serão removidas apenas da base tratada, mantendo a base original preservada.

In [20]:
colunas_remover = [
    "revista",
    "receber_mais_atualizacoes_sobre_nossos_cursos",
    "atualizar_sobre_conteudo_de_supply_chain",
    "receber_atualizacoes_sobre_conteudo_de_marketing_digital",
    "concordo_em_pagar_o_valor_por_cheque",
    "busca",
    "artigo_de_jornal",
    "foruns_da_x_education",
    "jornal",
    "anuncio_digital",
    "por_recomendacoes",
    "nao_ligar"
]

df_tratado = df_tratado.drop(
    columns=[coluna for coluna in colunas_remover if coluna in df_tratado.columns]
)
print(f"Base original: {df.shape[0]} linhas e {df.shape[1]} colunas")
print(f"Base tratada: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas")

Base original: 9240 linhas e 37 colunas
Base tratada: 9240 linhas e 25 colunas


In [21]:
print(f"Total de linhas: {df_tratado.shape[0]}")
print(f"Total de colunas: {df_tratado.shape[1]}")
print(f"Total de valores nulos: {df_tratado.isnull().sum().sum()}")
print(f"Total de linhas duplicadas: {df_tratado.duplicated().sum()}")
print(f"Taxa geral de conversão: {df_tratado['convertido'].mean() * 100:.2f}%")
df_tratado.head()

Total de linhas: 9240
Total de colunas: 25
Total de valores nulos: 0
Total de linhas duplicadas: 0
Taxa geral de conversão: 38.54%


,id_do_prospect,numero_do_lead,origem_do_lead,fonte_do_lead,nao_enviar_e-mail,convertido,total_de_visitas,tempo_total_no_site,visualizacoes_de_pagina_por_visita,ultima_atividade,...,tags,qualidade_do_lead,perfil_do_lead,cidade,indice_de_atividade_assimetrica,indice_de_perfil_assimetrico,pontuacao_de_atividade_assimetrica,pontuacao_de_perfil_assimetrico,copia_gratuita_de_mastering_the_interview,ultima_atividade_relevante
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Chat Olark,Não,0,0.0,0,0.0,Página Visitada no Site,...,Interessado em outros cursos,Baixa Relevância,Não informado,Não informado,02.Médio,02.Médio,15.0,15.0,Não,Modificado
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,API,Busca Orgânica,Não,0,5.0,674,2.5,E-mail Aberto,...,Chamando,Não informado,Não informado,Não informado,02.Médio,02.Médio,15.0,15.0,Não,E-mail Aberto
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,Envio por Landing Page,Tráfego Direto,Não,1,2.0,1532,2.0,E-mail Aberto,...,Retornará após ler o e-mail,Talvez,Lead Potencial,Mumbai,02.Médio,01.Alto,14.0,20.0,Sim,E-mail Aberto
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc,660719,Envio por Landing Page,Tráfego Direto,Não,0,1.0,305,1.0,Inalcançável,...,Chamando,Incerto,Não informado,Mumbai,02.Médio,01.Alto,13.0,17.0,Não,Modificado
4,3256f628-e534-4826-9d63-4a8b88782852,660681,Envio por Landing Page,Google,Não,1,2.0,1428,1.0,Convertido em Lead,...,Retornará após ler o e-mail,Talvez,Não informado,Mumbai,02.Médio,01.Alto,15.0,18.0,Não,Modificado


In [22]:
df_tratado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9240 entries, 0 to 9239
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   id_do_prospect                             9240 non-null   object 
 1   numero_do_lead                             9240 non-null   int64  
 2   origem_do_lead                             9240 non-null   object 
 3   fonte_do_lead                              9240 non-null   object 
 4   nao_enviar_e-mail                          9240 non-null   object 
 5   convertido                                 9240 non-null   int64  
 6   total_de_visitas                           9240 non-null   float64
 7   tempo_total_no_site                        9240 non-null   int64  
 8   visualizacoes_de_pagina_por_visita         9240 non-null   float64
 9   ultima_atividade                           9240 non-null   object 
 10  pais                    

In [23]:
df_tratado.to_csv("leads_tratados_mvp1.csv", index=False, encoding="utf-8-sig")
from google.colab import files

files.download("leads_tratados_mvp1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>